# Arquitectura Medallón: Pipeline de Datos GastroData
Este notebook demuestra el flujo de datos desde la ingesta bruta hasta la generación de insights para el Dashboard.

**Etapas:**
1. **Bronce:** Ingesta de datos crudos (ventas sin procesar).
2. **Plata:** Limpieza de datos, manejo de nulos y tipado.
3. **Oro:** Agregaciones de negocio y tablas listas para el consumo del dashboard.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Crear carpetas si no existen
for layer in ['capa_bronce', 'capa_plata', 'capa_oro']:
    if not os.path.exists(layer):
        os.makedirs(layer)

## 1. Capa BRONCE: Ingesta Bruta
Simulamos la recepción de datos desde el sistema POS del restaurante.

In [ ]:
def generar_datos_bronce(n_transacciones=1000):
    platos = ['Hamburguesa Alpha', 'Pizza Data', 'Ensalada Neural', 'Tacos Logic', 'Sushi Flux']
    precios = [25000, 35000, 18000, 22000, 28000]
    
    data = {
        'id_transaccion': range(1, n_transacciones + 1),
        'fecha_hora': [datetime.now() - timedelta(minutes=np.random.randint(0, 43200)) for _ in range(n_transacciones)],
        'producto': [np.random.choice(platos) for _ in range(n_transacciones)],
        'cantidad': [np.random.randint(1, 4) for _ in range(n_transacciones)],
        'mesa': [np.random.randint(1, 21) for _ in range(n_transacciones)],
        'valor_pagado': [0] * n_transacciones # Se calculará en Plata
    }
    
    df_bronce = pd.DataFrame(data)
    df_bronce.to_csv('capa_bronce/ventas_raw.csv', index=False)
    return df_bronce

df_bronce = generar_datos_bronce()
print("Capa Bronce Creada: ", df_bronce.shape)
df_bronce.head()

## 2. Capa PLATA: Limpieza y Transformación
En esta etapa limpiamos los datos y calculamos valores base.

In [ ]:
def procesar_capa_plata():
    df = pd.read_csv('capa_bronce/ventas_raw.csv')
    
    # Mapeo de precios (Estandarización)
    precios_dict = {
        'Hamburguesa Alpha': 25000, 'Pizza Data': 35000, 
        'Ensalada Neural': 18000, 'Tacos Logic': 22000, 'Sushi Flux': 28000
    }
    
    # Limpieza y Cálculo
    df['precio_unitario'] = df['producto'].map(precios_dict)
    df['valor_total'] = df['precio_unitario'] * df['cantidad']
    df['costo_estimado'] = df['valor_total'] * 0.45
    df['margen_ganancia'] = df['valor_total'] - df['costo_estimado']
    
    # Convertir fechas
    df['fecha_hora'] = pd.to_datetime(df['fecha_hora'])
    df['fecha'] = df['fecha_hora'].dt.date
    
    df.to_csv('capa_plata/ventas_limpias.csv', index=False)
    return df

df_plata = procesar_capa_plata()
print("Capa Plata Creada: ", df_plata.shape)
df_plata.head()

## 3. Capa ORO: Insights de Negocio
Agregaciones listas para el Dashboard.

In [ ]:
def generar_capa_oro():
    df = pd.read_csv('capa_plata/ventas_limpias.csv')
    
    # Agregación por Producto (Rentabilidad)
    rentabilidad = df.groupby('producto').agg({
        'valor_total': 'sum',
        'margen_ganancia': 'sum',
        'cantidad': 'sum'
    }).reset_index().rename(columns={'valor_total': 'Ingresos_Totales'})
    
    # Agregación Diaria (Tendencia)
    tendencia = df.groupby('fecha').agg({
        'valor_total': 'sum',
        'margen_ganancia': 'sum'
    }).reset_index().rename(columns={'valor_total': 'Ingresos_Diarios'})
    
    # Guardar en carpeta final
    rentabilidad.to_csv('capa_oro/rentabilidad_producto.csv', index=False)
    tendencia.to_csv('capa_oro/ventas_diarias.csv', index=False)
    
    return rentabilidad, tendencia

df_oro_rent, df_oro_trend = generar_capa_oro()
print("Capa Oro Generada con Éxito.")
df_oro_rent.sort_values(by='Ingresos_Totales', ascending=False)